# Perceptron Implementation

Perceptron classification:
    Uses Big Five traits to predict Low/Medium/High CWB and OCB.

Algorithm idea:
    A perceptron is a simple linear classifier. It learns a weighted combination
    of predictors and updates weights when it misclassifies cases.

Evaluation metrics:
    - Higher accuracy
    - Higher precision
    - Higher recall
    - Higher macro F1

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Perceptron
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report, confusion_matrix
from sklearn.pipeline import Pipeline

# Converts continuous scores into three equally sized groups using tertiles
def make_tertiles(series):
    return pd.qcut(series, q=3, labels=["Low", "Medium", "High"])

# Perceptron classification function
def run_perceptron_classification(df, random_state=42):

    # Define Big Five personality traits as predictor variables
    big_five = [
        "Extraversion",
        "Agreeableness",
        "Conscientiousness",
        "Neuroticism",
        "Openness",
    ]
    # Define outcome variables to predict
    outcomes = ["CWB", "OCB"]
    # Create dictionary to store model results
    results = {}

    # Run a separate Perceptron classification model for each outcome
    for outcome in outcomes:
        data = df[big_five + [outcome]].dropna().copy()
        data[f"{outcome}_class"] = make_tertiles(data[outcome])

        X = data[big_five]
        y = data[f"{outcome}_class"]

        # Split dataset into training and testing sets (80% training, 20% testing)
        X_train, X_test, y_train, y_test = train_test_split(
            X,
            y,
            test_size=0.2,
            random_state=random_state,
            stratify=y
        )

        # Create machine learning pipeline
        model = Pipeline([
            ("scaler", StandardScaler()),
            ("perceptron", Perceptron(max_iter=1000, random_state=random_state))
        ])
        # Fit model and make predictions
        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)

        # Calculate evaluation metrics
        metrics = {
            "algorithm": "Perceptron",
            "outcome": outcome,
            "task": "classification",
            "accuracy": accuracy_score(y_test, y_pred),
            "precision_macro": precision_score(y_test, y_pred, average="macro", zero_division=0),
            "recall_macro": recall_score(y_test, y_pred, average="macro", zero_division=0),
            "f1_macro": f1_score(y_test, y_pred, average="macro", zero_division=0),
            "classification_report": classification_report(y_test, y_pred, zero_division=0),
            "confusion_matrix": confusion_matrix(y_test, y_pred),
            "model": model,
            
        }

        results[outcome] = metrics

    return results

## Import dataset and data preprocessing

Preprocess data to transform big five and organizational behavioral measures.

In [1]:
# import predetermined functions from src/
import sys
import os
sys.path.append(os.path.abspath("../../src"))
from ml_models.perceptron import run_perceptron_classification

# import and preprocess data
import json
from pathlib import Path
import pandas as pd

# Find repo root automatically
repo_root = Path.cwd().resolve()
while not (repo_root / "BFI_2_life_narative_metadata.json").exists():
    if repo_root == repo_root.parent:
        raise FileNotFoundError("Could not find BFI_2_life_narative_metadata.json")
    repo_root = repo_root.parent

# Load data
with open(repo_root / "BFI_2_life_narative_metadata.json", "r", encoding="utf-8") as f:
    metadata = json.load(f)

with open(repo_root / "BFI_2_life_narrative.json", "r", encoding="utf-8") as f:
    responses = json.load(f)

df = pd.DataFrame(responses)

# Reverse-code BFI items
reverse_items = metadata["reverse_code_items"]

for item in reverse_items:
    if item in df.columns:
        df[item] = 6 - df[item]

# Big Five trait means
traits = {
    key: value
    for key, value in metadata.items()
    if (
        isinstance(value, list)
        and value
        and isinstance(value[0], str)
        and value[0].startswith("Item")
        and not key.startswith(("Item", "Q", "CWB", "OCB"))
        and key != "reverse_code_items"
    )
}

for trait, items in traits.items():
    available_items = [item for item in items if item in df.columns]
    df[trait] = df[available_items].mean(axis=1)

# CWB and OCB means
cwb_cols = [f"CWB{i}" for i in range(1, 11) if f"CWB{i}" in df.columns]
ocb_cols = [f"OCB{i}" for i in range(1, 11) if f"OCB{i}" in df.columns]

df["CWB"] = df[cwb_cols].mean(axis=1)
df["OCB"] = df[ocb_cols].mean(axis=1)

# Targets
big_five = [
    "Extraversion",
    "Agreeableness",
    "Conscientiousness",
    "Neuroticism",
    "Openness",
]

print("✓ Data preprocessing complete")
print("Repo root:", repo_root)
print("df shape:", df.shape)
df.head()

✓ Data preprocessing complete
Repo root: /Users/cindy/cmor438_Spring2026/cmor438_Spring2026
df shape: (500, 138)


,ID,Gender,Race,Age,Q1,Q2,Q3,Q4,Q5,Q6,...,Intellectual_Curiosity,Aesthetic_Sensitivity,Creative_Imagination,Extraversion,Agreeableness,Conscientiousness,Neuroticism,Openness,CWB,OCB
0,1,Woman,White,22,"chicago, illinois; in a suburb near chicago. i...",i always worked very hard and did my best. i h...,i had multiple teachers that were influential ...,art or reading/writing. i am a really creative...,"math, because it doesn't always come as easily...",one of my heroes has always been my mom. she a...,...,4.50,4.75,5.00,2.416667,4.250000,3.083333,4.500000,4.750000,1.7,3.2
1,2,Woman,White,38,I am from VA and you? I grew up in TX and it w...,A very hardworking and a brilliant student in ...,I always liked my Mathematics teacher very muc...,I loved mathematics and biology as I loved to ...,Probably geography was my least favorite for l...,For me it was always Einstein as he was an awe...,...,3.50,3.75,4.25,4.583333,4.250000,4.250000,1.583333,3.833333,1.1,2.7
2,3,Woman,White,19,"I'm from Wichita,Kansas My life was great grow...",I was horrible in school. I was diagnosed with...,Oh yeah! I had a great teacher senior year of ...,My favorite subject in school was probably His...,I hated English. I was never good at it.,my heros were probably my parents. They were a...,...,3.50,4.25,4.50,3.916667,3.916667,2.500000,3.500000,4.083333,2.1,3.6
3,4,Man,White,21,"northbrook, IL; I grew up in the northern subu...",I was a great student. I was in Honors and AP ...,I definitely had influential teachers. My AP p...,Math because there was a definitive answer and...,History because it was so boring to me alwya,My mom for sure and brothers,...,3.50,3.00,3.25,4.666667,4.500000,3.083333,2.583333,3.250000,1.5,2.8
4,5,Woman,White,26,"I am from Denver Colorado, I grew up in Cherry...",I was a very bright student in school and ever...,My mathematics teacher was influtial in my pro...,Mathematics was my favorite subject because i ...,History was my worst subject because the class...,My parents were my heroes because they gave me...,...,3.75,3.25,3.25,2.916667,3.250000,3.583333,2.916667,3.416667,1.5,2.8


## Run perceptron

In [2]:
# define predictors and outcomes
predictors = [
    "Extraversion",
    "Agreeableness",
    "Conscientiousness",
    "Neuroticism",
    "Openness",
]
outcomes = ["CWB", "OCB"]

In [ ]:
# run perceptron classification
perceptron_results = run_perceptron_classification(
    df=df,
    predictors=predictors,
    outcomes=outcomes,
    test_size=0.2,
    random_state=42
)

In [4]:
# view metrics
for outcome, result in perceptron_results.items():
    print(f"\n{outcome}")
    print("Accuracy:", round(result["accuracy"], 4))
    print("Precision:", round(result["precision_macro"], 4))
    print("Recall:", round(result["recall_macro"], 4))
    print("Macro F1:", round(result["f1_macro"], 4))
    print(result["classification_report"])


CWB
Accuracy: 0.39
Precision: 0.3954
Recall: 0.3857
Macro F1: 0.3861
              precision    recall  f1-score   support

        High       0.29      0.26      0.27        31
         Low       0.55      0.44      0.49        36
      Medium       0.35      0.45      0.39        33

    accuracy                           0.39       100
   macro avg       0.40      0.39      0.39       100
weighted avg       0.40      0.39      0.39       100


OCB
Accuracy: 0.38
Precision: 0.365
Recall: 0.372
Macro F1: 0.3649
              precision    recall  f1-score   support

        High       0.38      0.47      0.42        32
         Low       0.45      0.45      0.45        38
      Medium       0.27      0.20      0.23        30

    accuracy                           0.38       100
   macro avg       0.37      0.37      0.36       100
weighted avg       0.37      0.38      0.37       100



## Results evaluation

The perceptron model was used to classify participants into **Low, Medium, or High** levels of CWB and OCB based on the Big Five personality traits.

### CWB Classification

| Metric | Value |
|---|---:|
| Accuracy | 0.39 |
| Precision (Macro) | 0.395 |
| Recall (Macro) | 0.386 |
| F1 (Macro) | 0.386 |

The perceptron achieved **39% accuracy** when classifying CWB levels. Because random guessing across three classes would be approximately 33%, the model performed only modestly above chance.

Class-level performance showed that the model predicted the **Low CWB** group best (F1 = 0.49), while performance was weaker for the **Medium** group (F1 = 0.39) and especially the **High CWB** group (F1 = 0.27).

The confusion matrix suggests that many High CWB cases were misclassified as Medium, indicating difficulty separating individuals with elevated counterproductive behavior from those in the middle range.

### OCB Classification

| Metric | Value |
|---|---:|
| Accuracy | 0.38 |
| Precision (Macro) | 0.365 |
| Recall (Macro) | 0.372 |
| F1 (Macro) | 0.365 |

For OCB, the perceptron achieved **38% accuracy**, again only slightly above chance.

The model performed best for the **Low OCB** group (F1 = 0.45) and reasonably for the **High OCB** group (F1 = 0.42), but struggled with the **Medium OCB** group (F1 = 0.23).

This pattern suggests that extreme groups were easier to identify than individuals with moderate levels of citizenship behavior.

### Interpretation

Overall, the perceptron showed limited classification ability. As a simple linear classifier, it captured some signal between personality and workplace behavior, but performance was relatively weak and inconsistent across classes.